## ↓のデータフレーム変換関数を定義したセルを実行済みの状態で進めてください

In [ ]:
import requests
import time
import pandas as pd

def gen_boj_dataframe(
    db:str,
    codes:list[str],
    startdate:str,
    enddate:str) -> pd.DataFrame:
    url = "https://www.stat-search.boj.or.jp/api/v1/getDataCode"
    params = {
        "DB": db,
        "CODE": ",".join(codes),
        "FORMAT": "JSON",
        "LANG":"JP",
        "STARTDATE": startdate,
        "ENDDATE": enddate
        }

    # ページネーションで最後のページを取得するまで、繰り返し処理を実行
    # Web APIからの取得データを格納
    all_results = []

    while True:
        # Web APIへのリクエストと例外処理。エラー時は空のデータフレームを返して終了
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            response_data = response.json()
        except requests.exceptions.Timeout:
            print("Web APIから30秒以内に応答がありませんでした。")
            return pd.DataFrame()
        except requests.exceptions.ConnectionError:
            print("Web APIに接続できませんでした。")
            return pd.DataFrame()
        except requests.exceptions.HTTPError as error:
            print(f"HTTPエラーが発生しました: {error}")

            content_type = response.headers.get("Content-Type", "")
            if "application/json" in content_type:
                error_data = response.json()
                print(error_data["MESSAGE"])
            return pd.DataFrame()
        except requests.exceptions.JSONDecodeError:
            print("レスポンスをJSONとして読み込めませんでした。")
            return pd.DataFrame()
        else:
            if response_data["MESSAGEID"] == "M181030I":
                print(response_data["MESSAGE"])
                return pd.DataFrame()

            all_results.extend(response_data["RESULTSET"])
            next_position = response_data["NEXTPOSITION"]
            if next_position is None:
                break

            params["STARTPOSITION"] = next_position
            time.sleep(1)

    # 取得結果をデータフレームに変換
    df = pd.DataFrame(all_results)
    result_df = pd.DataFrame()

    for i in range(len(df)):
        values_df = pd.DataFrame(df.loc[i,"VALUES"])
        values_df = values_df.assign(
            SERIES_CODE=df.loc[i, "SERIES_CODE"],
            NAME_OF_TIME_SERIES_J=df.loc[i, "NAME_OF_TIME_SERIES_J"],
            CATEGORY_J=df.loc[i, "CATEGORY_J"]
        )
        result_df = pd.concat([result_df, values_df])

    result_df = result_df[["SERIES_CODE","NAME_OF_TIME_SERIES_J","CATEGORY_J","SURVEY_DATES","VALUES"]]

    result_df = result_df.reset_index(drop=True)

    result_df = result_df.astype({'SURVEY_DATES': str})

    return result_df

## 日銀短観をヒートマップで表示し、業種・企業規模別の景況感を見てみよう

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 系列コードの文字位置ごとの内容を辞書化
## 業種
industry_dict = {
    "0000": "全産業","1000": "製造業","1020": "繊維","1030": "木材・木製品","1050": "紙・パルプ","1060": "化学","1070": "石油・石炭製品","1100": "窯業・土石製品",
    "1110": "鉄鋼","1120": "非鉄金属","1010": "食料品","1130": "金属製品","1149": "はん用・生産用・業務用機械","1141": "はん用機械","1142": "生産用機械","1143": "業務用機械",
    "1150": "電気機械","1160": "輸送用機械","1185": "造船・重機、その他輸送用機械","1180": "自動車","1500": "その他製造業","3030": "素材業種","3040": "加工業種","2000": "非製造業",
    "2011": "建設","2019": "不動産・物品賃貸","2012": "不動産","2090": "物品賃貸","2020": "卸売・小売","2021": "卸売","2024": "小売","2040": "運輸・郵便",
    "2059": "情報通信","2050": "通信","2051": "情報サービス","2052": "その他情報通信","2060": "電気・ガス","2081": "対事業所サービス","2082": "対個人サービス","2100": "宿泊・飲食サービス",
    "2500": "鉱業・採石業・砂利採取業",
}

## 企業規模
company_size_dict = {
    "0":"0_全規模合計","1":"1_大企業","2":"2_中堅企業","3":"3_中小企業",
}

## 実績・予測
actual_forecast_dict = {
    "0":"実績","1":"予測"
}

# 系列コードのリスト化
url = "https://www.stat-search.boj.or.jp/api/v1/getMetadata"

response_co = requests.get(url, params={"DB":"CO"})

data_co = response_co.json()

co_series_data = [
    {
        "SERIES_CODE": item["SERIES_CODE"],
        "NAME_OF_TIME_SERIES_J": item["NAME_OF_TIME_SERIES_J"],
        "CATEGORY_J": item["CATEGORY_J"],
    }
    for item in data_co.get("RESULTSET", [])
    if item.get("SERIES_CODE")
]

co_series_codes_df = pd.DataFrame(co_series_data)

# 業況判断D.Iの系列情報を抽出
co_di_series_codes_df = co_series_codes_df.query('SERIES_CODE.str.slice(9, 13) == "601G"')

# 業種を紐づけし、紐づけできないもの（欠損値）を除外
co_di_series_codes_df["業種"] = co_di_series_codes_df["SERIES_CODE"].str[5:9].map(industry_dict)
co_di_series_codes_df = co_di_series_codes_df.dropna(subset=["業種"])

# 重複を除去してリスト化
co_di_seiries_codes = co_di_series_codes_df["SERIES_CODE"].drop_duplicates().tolist()

# リストから統計値データを取得
co_data_df = gen_boj_dataframe("CO", co_di_seiries_codes, "202602","202602")

# 文字列情報から業種・企業規模・実績予測のデータを付与
co_data_df["業種"] = co_data_df["SERIES_CODE"].str[5:9].map(industry_dict)
co_data_df["企業規模"] = co_data_df["SERIES_CODE"].str[16].map(company_size_dict)
co_data_df["実績予測"] = co_data_df["SERIES_CODE"].str[15].map(actual_forecast_dict)

# 実績/予測別のデータフレームを作成
actual_heatmap_df = co_data_df.query('実績予測 == "実績"')
forecast_heatmap_df = co_data_df.query('実績予測 == "予測"')

# ヒートマップ作成のための、必要なデータのみ残した横長形式のデータフレームへ変換
actual_heatmap_data = actual_heatmap_df.pivot(
    index="業種",
    columns="企業規模",
    values="VALUES",
)

forecast_heatmap_data = forecast_heatmap_df.pivot(
    index="業種",
    columns="企業規模",
    values="VALUES",
)

# ヒートマップの作成
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    subplot_titles=("実績（最近）", "予測（先行き）"),
)

fig.add_trace(
    go.Heatmap(
        z=actual_heatmap_data,
        x=actual_heatmap_data.columns,
        y=actual_heatmap_data.index,
        coloraxis="coloraxis",
        texttemplate="%{z:.0f}",
        ),
    row=1,
    col=1,
    )

fig.add_trace(
    go.Heatmap(
        z=forecast_heatmap_data,
        x=forecast_heatmap_data.columns,
        y=forecast_heatmap_data.index,
        coloraxis="coloraxis",
        texttemplate="%{z:.0f}",
        ),
    row=1,
    col=2,
    )

# 両図で色の尺度を共有し、同じ値を同じ色で表示する
fig.update_layout(
    title="業種・企業規模別の業況判断D.I.",
    coloraxis={
        "colorscale": "RdBu_r",
        "cmid": 0,
        "colorbar": {"title": "D.I.（％ポイント）"},
    },
    height=1000,
)

fig.update_xaxes(title_text="企業規模")
fig.update_yaxes(title_text="業種", row=1, col=1)

fig.show()

## 2026年第1四半期と第2四半期の、実績予想の差分のヒートマップを並べて表示

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 系列コードの文字位置ごとの内容を辞書化
## 業種
industry_dict = {
    "0000": "全産業","1000": "製造業","1020": "繊維","1030": "木材・木製品","1050": "紙・パルプ","1060": "化学","1070": "石油・石炭製品","1100": "窯業・土石製品",
    "1110": "鉄鋼","1120": "非鉄金属","1010": "食料品","1130": "金属製品","1149": "はん用・生産用・業務用機械","1141": "はん用機械","1142": "生産用機械","1143": "業務用機械",
    "1150": "電気機械","1160": "輸送用機械","1185": "造船・重機、その他輸送用機械","1180": "自動車","1500": "その他製造業","3030": "素材業種","3040": "加工業種","2000": "非製造業",
    "2011": "建設","2019": "不動産・物品賃貸","2012": "不動産","2090": "物品賃貸","2020": "卸売・小売","2021": "卸売","2024": "小売","2040": "運輸・郵便",
    "2059": "情報通信","2050": "通信","2051": "情報サービス","2052": "その他情報通信","2060": "電気・ガス","2081": "対事業所サービス","2082": "対個人サービス","2100": "宿泊・飲食サービス",
    "2500": "鉱業・採石業・砂利採取業",
}

## 企業規模
company_size_dict = {
    "0":"0_全規模合計","1":"1_大企業","2":"2_中堅企業","3":"3_中小企業",
}

## 実績・予測
actual_forecast_dict = {
    "0":"実績","1":"予測"
}

# 系列コードのリスト化
url = "https://www.stat-search.boj.or.jp/api/v1/getMetadata"

response_co = requests.get(url, params={"DB":"CO"})

data_co = response_co.json()

co_series_data = [
    {
        "SERIES_CODE": item["SERIES_CODE"],
        "NAME_OF_TIME_SERIES_J": item["NAME_OF_TIME_SERIES_J"],
        "CATEGORY_J": item["CATEGORY_J"],
    }
    for item in data_co.get("RESULTSET", [])
    if item.get("SERIES_CODE")
]

co_series_codes_df = pd.DataFrame(co_series_data)

# 業況判断D.Iの系列情報を抽出
co_di_series_codes_df = co_series_codes_df.query('SERIES_CODE.str.slice(9, 13) == "601G"')

# 業種を紐づけし、紐づけできないもの（欠損値）を除外
co_di_series_codes_df["業種"] = co_di_series_codes_df["SERIES_CODE"].str[5:9].map(industry_dict)
co_di_series_codes_df = co_di_series_codes_df.dropna(subset=["業種"])

# 重複を除去してリスト化
co_di_seiries_codes = co_di_series_codes_df["SERIES_CODE"].drop_duplicates().tolist()

# リストから統計値データを取得
co_data_df = gen_boj_dataframe("CO", co_di_seiries_codes, "202504","202602")

# 文字列情報から業種・企業規模・実績予測のデータを付与
co_data_df["業種"] = co_data_df["SERIES_CODE"].str[5:9].map(industry_dict)
co_data_df["企業規模"] = co_data_df["SERIES_CODE"].str[16].map(company_size_dict)
co_data_df["実績予測"] = co_data_df["SERIES_CODE"].str[15].map(actual_forecast_dict)

# 実績と予測を分け、行を業種・企業規模、列を期間とする表に変換
actual_df = co_data_df.query('実績予測 == "実績"').pivot(
    index=["業種", "企業規模"],
    columns="SURVEY_DATES",
    values="VALUES",
)

forecast_df = co_data_df.query('実績予測 == "予測"').pivot(
    index=["業種", "企業規模"],
    columns="SURVEY_DATES",
    values="VALUES",
)

# 同じ業種・企業規模の当期実績から前期予測を引く
# 業種・企業規模の列を対応づけて計算
actual_df["第1四半期ギャップ"] = actual_df["202601"] - forecast_df["202504"]
actual_df["第2四半期ギャップ"] = actual_df["202602"] - forecast_df["202601"]
actual_forecast_df = actual_df.reset_index()

# ヒートマップ用に、行を業種、列を企業規模とする表へ変換
first_quarter_heatmap_data = actual_forecast_df.pivot(
    index="業種",
    columns="企業規模",
    values="第1四半期ギャップ",
)

second_quarter_heatmap_data = actual_forecast_df.pivot(
    index="業種",
    columns="企業規模",
    values="第2四半期ギャップ",
)

# ヒートマップの作成
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    subplot_titles=("2026年第1四半期", "2026第2四半期"),
)

fig.add_trace(
    go.Heatmap(
        z=first_quarter_heatmap_data,
        x=first_quarter_heatmap_data.columns,
        y=first_quarter_heatmap_data.index,
        coloraxis="coloraxis",
        texttemplate="%{z:.0f}",
        ),
    row=1,
    col=1,
    )

fig.add_trace(
    go.Heatmap(
        z=second_quarter_heatmap_data,
        x=second_quarter_heatmap_data.columns,
        y=second_quarter_heatmap_data.index,
        coloraxis="coloraxis",
        texttemplate="%{z:.0f}",
        ),
    row=1,
    col=2,
    )

# 両図で色の尺度を共有し、同じ値を同じ色で表示する
fig.update_layout(
    title="業種・企業規模別の業況判断D.I.（当期実績 − 前期予測）",
    coloraxis={
        "colorscale": "RdBu_r",
        "cmid": 0,
        "colorbar": {"title": "当期実績 − 前期予測（％ポイント）"},
    },
    height=1000,
)

fig.update_xaxes(title_text="企業規模")
fig.update_yaxes(title_text="業種", row=1, col=1)

fig.show()